# CyberSec-FT-LLM — Unsloth QLoRA Fine-Tuning
Fine-tunes **Phi-3.5-mini-Instruct** on CVE/Exploit security data using Unsloth.

**Before running:**
1. Runtime → Change runtime type → **GPU (T4 or A100)**
2. Upload `dataset/` to Google Drive at: `My Drive/CyberSec-FT-LLM/dataset/`
3. Run cells in order

In [ ]:
# Cell 1: Install Dependencies
!pip install -q unsloth trl>=0.9.0 transformers>=4.45.0 datasets accelerate peft bitsandbytes pyyaml rouge-score
print('Dependencies installed')

In [ ]:
# Cell 2: Check GPU
import torch
print(f'CUDA: {torch.cuda.is_available()}')
print(f'GPU : {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
# Cell 3: Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

import os
DATASET_DIR = '/content/drive/MyDrive/CyberSec-FT-LLM/dataset'
for f in ['train.jsonl', 'val.jsonl', 'test.jsonl']:
    path = os.path.join(DATASET_DIR, f)
    if os.path.exists(path):
        size_mb = os.path.getsize(path) / 1e6
        print(f'  {f}: {size_mb:.1f} MB')
    else:
        print(f'  {f}: NOT FOUND - upload to Drive first!')

In [ ]:
# Cell 4: Clone Repo
import os

REPO_URL = 'https://github.com/Mohamedabul/CyberSec-FT-LLM.git'
REPO_DIR = '/content/CyberSec-FT-LLM'

if os.path.exists(REPO_DIR):
    print('Repo exists - pulling latest...')
    !cd {REPO_DIR} && git pull
else:
    !git clone {REPO_URL} {REPO_DIR}

os.chdir(REPO_DIR)
print(f'Working directory: {os.getcwd()}')
print('Contents:', os.listdir('.'))

In [ ]:
# Cell 5: Link Dataset from Drive
import os

DATASET_SRC = '/content/drive/MyDrive/CyberSec-FT-LLM/dataset'
DATASET_DST = '/content/CyberSec-FT-LLM/dataset'
os.makedirs(DATASET_DST, exist_ok=True)

for f in ['train.jsonl', 'val.jsonl', 'test.jsonl']:
    src = os.path.join(DATASET_SRC, f)
    dst = os.path.join(DATASET_DST, f)
    if os.path.exists(dst) or os.path.islink(dst):
        os.remove(dst)
    if os.path.exists(src):
        os.symlink(src, dst)
        print(f'  Linked: {f}')
    else:
        print(f'  MISSING: {f} not found in Drive!')

In [ ]:
# Cell 6: Configure Training Paths
import yaml, os

config_path = '/content/CyberSec-FT-LLM/configs/training_config.yaml'

# Verify it exists
if not os.path.exists(config_path):
    raise FileNotFoundError(f'Config not found: {config_path}\nCheck Cell 4 cloned successfully.')

with open(config_path) as f:
    cfg = yaml.safe_load(f)

cfg['colab']['output_dir']   = '/content/drive/MyDrive/CyberSec-FT-LLM/models/adapter'
cfg['colab']['dataset_path'] = '/content/CyberSec-FT-LLM/dataset'

with open(config_path, 'w') as f:
    yaml.dump(cfg, f, default_flow_style=False)

print('Config updated:')
print(f"  output_dir  : {cfg['colab']['output_dir']}")
print(f"  dataset_path: {cfg['colab']['dataset_path']}")
os.makedirs(cfg['colab']['output_dir'], exist_ok=True)

In [ ]:
# Cell 7: Run Training (auto-resumes from checkpoint if stopped)
import subprocess, sys

result = subprocess.run(
    [sys.executable, '/content/CyberSec-FT-LLM/training/unsloth/train_unsloth.py'],
    cwd='/content/CyberSec-FT-LLM/training/unsloth'
)
print('Training complete!' if result.returncode == 0 else f'Error (code {result.returncode})')

In [ ]:
# Cell 8: Quick Inference Test
import sys, torch
sys.path.insert(0, '/content/CyberSec-FT-LLM/training/unsloth')

from model_loader_unsloth import load_for_inference

ADAPTER_DIR = '/content/drive/MyDrive/CyberSec-FT-LLM/models/adapter'
model, tokenizer = load_for_inference(ADAPTER_DIR, max_seq_length=512)

instruction = 'Analyze the following CVE and provide a structured vulnerability report.'
context = (
    'CVE ID: CVE-2021-44228\n'
    'Description: Apache Log4j2 JNDI RCE vulnerability (Log4Shell).\n'
    'CVSS v3: 10.0 CRITICAL | Attack Vector: NETWORK | CWE: CWE-917'
)
messages = [{'role': 'user', 'content': f'{instruction}\n\n{context}'}]
prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(prompt, return_tensors='pt').to(model.device)

with torch.no_grad():
    output = model.generate(**inputs, max_new_tokens=400, temperature=0.7, do_sample=True)

generated = output[0][inputs['input_ids'].shape[1]:]
print(tokenizer.decode(generated, skip_special_tokens=True))

In [ ]:
# Cell 9: Verify Saved Files on Drive
import os
for folder in [
    '/content/drive/MyDrive/CyberSec-FT-LLM/models/adapter',
    '/content/drive/MyDrive/CyberSec-FT-LLM/models/merged',
]:
    if os.path.exists(folder):
        files = os.listdir(folder)
        size = sum(os.path.getsize(os.path.join(folder, f)) for f in files) / 1e6
        print(f'{folder}\n  {len(files)} files | {size:.0f} MB')
    else:
        print(f'Not found: {folder}')